In [12]:
EMBEDDING_MODEL = 'nomic-embed-text'
LLM_MODEL = 'llama3.1'
from typing import List, Dict, Any

In [13]:
import ollama


def llm_response(prompt: str) -> str:
    response = ollama.chat(model=LLM_MODEL, messages=[{"role": "user", "content": prompt}])

    if isinstance(response, dict):
        message = response.get("message", {})
        if isinstance(message, dict):
            return str(message.get("content", ""))

    if hasattr(response, "message"):
        message_obj = getattr(response, "message")
        if isinstance(message_obj, dict):
            return str(message_obj.get("content", ""))
        if hasattr(message_obj, "content"):
            return str(message_obj.content)

    return str(response)


def embed_text(text: str) -> List[float]:
    response = ollama.embed(model=EMBEDDING_MODEL, input=text)

    if isinstance(response, dict):
        embedding = response.get("embedding")
        if isinstance(embedding, list) and embedding:
            if isinstance(embedding[0], (int, float)):
                return [float(x) for x in embedding]
            if isinstance(embedding[0], list) and embedding[0]:
                return [float(x) for x in embedding[0]]

        embeddings = response.get("embeddings")
        if isinstance(embeddings, list) and embeddings:
            first = embeddings[0]
            if isinstance(first, list):
                return [float(x) for x in first]

    if hasattr(response, "embeddings"):
        embeddings_attr = getattr(response, "embeddings")
        if isinstance(embeddings_attr, list) and embeddings_attr:
            first = embeddings_attr[0]
            if isinstance(first, list):
                return [float(x) for x in first]

    raise ValueError("Could not parse embedding response from ollama.embed")

In [14]:
import re
from pathlib import Path
from typing import List, Dict, Any, Tuple

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter


def _dedupe_keep_order(items: List[Any]) -> List[Any]:
    seen = set()
    out: List[Any] = []
    for item in items:
        if item not in seen:
            seen.add(item)
            out.append(item)
    return out


def _build_document_text_and_page_spans(pages: List[Dict[str, Any]]) -> Tuple[str, List[Dict[str, int]]]:
    parts: List[str] = []
    spans: List[Dict[str, int]] = []
    cursor = 0

    for i, page in enumerate(pages):
        text = str(page.get("text", "")).strip()
        if not text:
            continue

        if parts:
            parts.append("\n\n")
            cursor += 2

        start = cursor
        parts.append(text)
        cursor += len(text)

        page_raw = page.get("page", i + 1)
        try:
            page_num = int(page_raw)
        except (TypeError, ValueError):
            page_num = i + 1

        spans.append({"start": start, "end": cursor, "page": page_num})

    return "".join(parts), spans


def split_numbered_sections_with_hierarchy(text: str) -> List[Dict[str, Any]]:
    """
    Split by numbered headings and keep hierarchical header context.
    Heading examples:
    1. Title
    3.1 Subtitle
    3.2.1 Sub-subtitle
    """
    heading_pattern = re.compile(r"(?m)^\s*(\d+(?:\.\d+)*)\.?\s+(.+?)\s*$")
    matches = list(heading_pattern.finditer(text))

    if not matches:
        cleaned = text.strip()
        return ([{"headers": [], "part": "Full Document", "body": cleaned, "start": 0, "end": len(cleaned)}] if cleaned else [])

    sections: List[Dict[str, Any]] = []
    stack: Dict[int, str] = {}

    for i, match in enumerate(matches):
        number = match.group(1).strip()
        title = match.group(2).strip()
        level = len(number.split("."))
        current_header = f"{number}. {title}"

        for k in list(stack.keys()):
            if k >= level:
                del stack[k]
        stack[level] = current_header

        headers = [stack[k] for k in sorted(stack.keys())]
        start = match.end()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
        body = text[start:end].strip()

        if not body:
            continue

        sections.append(
            {
                "headers": headers,
                "part": current_header,
                "body": body,
                "start": start,
                "end": end,
            }
        )

    return sections


def merge_small_sections(
    sections: List[Dict[str, Any]],
    min_section_chars: int = 300,
    max_merge_headers: int = 3,
) -> List[Dict[str, Any]]:
    """Merge tiny sections with the next one so chunks are more informative."""
    if not sections:
        return []

    merged: List[Dict[str, Any]] = []
    i = 0
    while i < len(sections):
        current = {
            "headers": list(sections[i].get("headers", [])),
            "part": sections[i].get("part", "Full Document"),
            "body": sections[i].get("body", "").strip(),
            "start": int(sections[i].get("start", 0)),
            "end": int(sections[i].get("end", 0)),
        }

        while len(current["body"]) < min_section_chars and i + 1 < len(sections):
            nxt = sections[i + 1]
            nxt_body = str(nxt.get("body", "")).strip()
            if nxt_body:
                current["body"] = (current["body"] + "\n\n" + nxt_body).strip()

            merged_headers = _dedupe_keep_order(current.get("headers", []) + nxt.get("headers", []))
            current["headers"] = merged_headers[-max_merge_headers:] if merged_headers else []
            if current["headers"]:
                current["part"] = current["headers"][-1]

            current["end"] = int(nxt.get("end", current["end"]))
            i += 1

        if current["body"]:
            merged.append(current)
        i += 1

    return merged


def _pages_for_span(start: int, end: int, page_spans: List[Dict[str, int]]) -> List[int]:
    pages: List[int] = []
    for span in page_spans:
        overlaps = span["start"] < end and start < span["end"]
        if overlaps:
            pages.append(int(span["page"]))
    return _dedupe_keep_order(pages)


def chunk_raw_pdfs(
    raw_dir: str = "../../data/raw",
    chunk_size: int = 1000,
    chunk_overlap: int = 150,
    min_section_chars: int = 300,
    max_header_depth_in_metadata: int = 3,
) -> List[Dict[str, Any]]:
    """
    Build chunks per full PDF (not per page), then attach page spans for attribution.
    This preserves section continuity while keeping course/part metadata clean for retrieval.
    """
    raw_path = Path(raw_dir).resolve()
    pdf_files = sorted(raw_path.glob("*.pdf"))

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ". ", " ", ""],
    )

    chunks: List[Dict[str, Any]] = []

    for pdf_path in pdf_files:
        course_title = pdf_path.stem.replace("_", " ").replace("-", " ").strip()
        docs = PyPDFLoader(str(pdf_path)).load()

        page_rows: List[Dict[str, Any]] = []
        for idx, doc in enumerate(docs, start=1):
            page_text = str(doc.page_content or "").strip()
            if not page_text:
                continue

            raw_page = (doc.metadata or {}).get("page", idx - 1)
            try:
                page_num = int(raw_page) + 1
            except (TypeError, ValueError):
                page_num = idx

            page_rows.append({"page": page_num, "text": page_text})

        if not page_rows:
            continue

        full_text, page_spans = _build_document_text_and_page_spans(page_rows)
        raw_sections = split_numbered_sections_with_hierarchy(full_text)
        sections = merge_small_sections(raw_sections, min_section_chars=min_section_chars)

        for section_idx, section in enumerate(sections, start=1):
            body = str(section.get("body", "")).strip()
            if not body:
                continue

            headers = section.get("headers", [])
            headers = headers[-max_header_depth_in_metadata:] if headers else []
            header_path = " > ".join(headers) if headers else ""
            part = headers[-1] if headers else str(section.get("part", "Full Document"))

            section_start = int(section.get("start", 0))
            section_end = int(section.get("end", section_start + len(body)))
            pages = _pages_for_span(section_start, section_end, page_spans)
            page_start = pages[0] if pages else None
            page_end = pages[-1] if pages else None

            context_prefix = header_path if header_path else part
            text_for_chunking = f"{context_prefix}\n{body}".strip()
            sub_chunks = splitter.split_text(text_for_chunking) or [text_for_chunking]

            for chunk_idx, sub_text in enumerate(sub_chunks, start=1):
                chunk_id = f"{course_title}::s{section_idx}::c{chunk_idx}"
                chunks.append(
                    {
                        "id": chunk_id,
                        "text": sub_text,
                        "metadata": {
                            "course": course_title,
                            "part": part,
                            "header_path": header_path,
                            "source": str(pdf_path),
                            "page_start": page_start,
                            "page_end": page_end,
                            "section_index": section_idx,
                            "chunk_index": chunk_idx,
                        },
                    }
                )

    return chunks


# Run
all_chunks = chunk_raw_pdfs(raw_dir="../../data/raw")
print(f"Total chunks created: {len(all_chunks)}")
print(all_chunks[:2])

Total chunks created: 609
[{'id': 'Classification metrics::s1::c1', 'text': "1. Accuracy\nAccuracy is a fundamental metric used for evaluating the performance of a classification model. \nIt tells us the proportion of correct predictions made by the model out of all predictions.  \nWhile accuracy provides a quick snapshot, it can be misleading in cases of imbalanced datasets. \nFor example, in a dataset with 90% class A and 10% class B, a model predicting only class A will \nstill achieve 90% accuracy but it will fail to identify any class B instances.  \nAccuracy is good but it gives a False Positive sense of achieving high accuracy. The problem \narises due to the possibility of misclassification of minor class samples being very high.  \nAccuracy = TP+TN/total Number of predictions  \nTP: True positive  \nTN: True Negative  \nFP: False positive  \nFN: False Negative 2. \nPrecision  \nIt measures how many of the positive predictions made by the model are actually correct. It's \nusef

In [15]:
import pandas as pd

df = pd.DataFrame(all_chunks)
display(df.head())
df["course"] = df["metadata"].apply(lambda x: x["course"])
df["part"] = df["metadata"].apply(lambda x: x["part"])

print("Total chunks:", len(df))
print("Unique courses:", df["course"].nunique())
print("Unique parts:", df["part"].nunique())

,id,text,metadata
0,Classification metrics::s1::c1,1. Accuracy\nAccuracy is a fundamental metric ...,"{'course': 'Classification metrics', 'part': '..."
1,Classification metrics::s1::c2,useful when the cost of false positives is hig...,"{'course': 'Classification metrics', 'part': '..."
2,Classification metrics::s2::c1,4. F1 Score\nThe F1 Score is the harmonic mean...,"{'course': 'Classification metrics', 'part': '..."
3,Classification metrics::s3::c1,5. Area Under Curve (AUC) and ROC Curve\nIt is...,"{'course': 'Classification metrics', 'part': '..."
4,Classification metrics::s3::c2,"cases, how many did the model correctly identi...","{'course': 'Classification metrics', 'part': '..."


Total chunks: 609
Unique courses: 28
Unique parts: 395


In [16]:
df[df["text"].str.contains("Optimization Algorithm", case=False, na=False)]

,id,text,metadata,course,part
237,Forward and Backward propagation::s5::c2,"After forward propagation, the network evaluat...","{'course': 'Forward and Backward propagation',...",Forward and Backward propagation,2. Backpropagation
238,Forward and Backward propagation::s5::c3,using an optimization algorithm like stochasti...,"{'course': 'Forward and Backward propagation',...",Forward and Backward propagation,2. Backpropagation
439,Loss functions::s2::c1,1. Regression Loss Functions\nThese are used w...,"{'course': 'Loss functions', 'part': '1. Regre...",Loss functions,1. Regression Loss Functions
440,Loss functions::s2::c2,zero which can cause issues for some optimizat...,"{'course': 'Loss functions', 'part': '1. Regre...",Loss functions,1. Regression Loss Functions
486,Optimization algorithm::s1::c1,1. What is Optimization Algorithm\nIn machine ...,"{'course': 'Optimization algorithm', 'part': '...",Optimization algorithm,1. What is Optimization Algorithm


In [17]:
text = df[(df['course']=="sql part1") | (df['course']=="sql part2") | (df['course']=='sql part3')]['text']
with open("sql_combined.txt", "w", encoding="utf-8") as f:
    f.write("\n*\n".join(text.tolist()))

In [18]:
from chromadb import PersistentClient
from chromadb.utils import embedding_functions


def _sanitize_for_chroma_metadata(meta: Dict[str, Any]) -> Dict[str, Any]:
    clean: Dict[str, Any] = {}
    for k, v in (meta or {}).items():
        if isinstance(v, (str, int, float, bool)) or v is None:
            clean[k] = v
        elif isinstance(v, list):
            clean[k] = " | ".join(str(x) for x in v)
        else:
            clean[k] = str(v)
    return clean


embedding_fn = embedding_functions.OllamaEmbeddingFunction(
    model_name=EMBEDDING_MODEL,
    url="http://localhost:11434/api/embeddings",
)
client = PersistentClient(path="../vectordb/chroma_db")
collection = client.get_or_create_collection(name="course_chunks", embedding_function=embedding_fn)

chunk_ids = df["id"].astype(str).tolist() if "id" in df.columns else df.index.astype(str).tolist()
sanitized_metadatas = [_sanitize_for_chroma_metadata(m) for m in df["metadata"].tolist()]

collection.upsert(
    ids=chunk_ids,
    documents=df["text"].astype(str).tolist(),
    metadatas=sanitized_metadatas,
)

print("Indexed chunks:", collection.count())

Indexed chunks: 609


In [19]:
# Reset all Chroma collections and rebuild vector DB from current df.
import gc
import shutil
import time
from pathlib import Path
from chromadb import PersistentClient
from chromadb.utils import embedding_functions

CHROMA_PATH = Path("../vectordb/chroma_db")


def _sanitize_for_chroma_metadata(meta: Dict[str, Any]) -> Dict[str, Any]:
    clean: Dict[str, Any] = {}
    for k, v in (meta or {}).items():
        if isinstance(v, (str, int, float, bool)) or v is None:
            clean[k] = v
        elif isinstance(v, list):
            clean[k] = " | ".join(str(x) for x in v)
        else:
            clean[k] = str(v)
    return clean


def _delete_all_collections(active_client):
    for col in active_client.list_collections():
        name = col.name if hasattr(col, "name") else str(col)
        active_client.delete_collection(name=name)


def _safe_rmtree(path: Path, retries: int = 4, base_sleep: float = 0.5) -> bool:
    for attempt in range(1, retries + 1):
        try:
            shutil.rmtree(path)
            return True
        except Exception as e:
            if attempt == retries:
                print(f"Folder cleanup warning: {e}")
                return False
            time.sleep(base_sleep * attempt)
    return False


# 1) Best-effort delete all collections via current client.
try:
    _delete_all_collections(client)
except Exception as e:
    print(f"Collection cleanup warning: {e}")

# 2) Release handles and try full on-disk reset.
try:
    del collection
except Exception:
    pass

try:
    del embedding_fn
except Exception:
    pass

try:
    del client
except Exception:
    pass

gc.collect()
time.sleep(1.0)

if CHROMA_PATH.exists():
    _safe_rmtree(CHROMA_PATH)

CHROMA_PATH.mkdir(parents=True, exist_ok=True)

# 3) Recreate client and ensure collections are empty.
embedding_fn = embedding_functions.OllamaEmbeddingFunction(
    model_name=EMBEDDING_MODEL,
    url="http://localhost:11434/api/embeddings",
)
client = PersistentClient(path=str(CHROMA_PATH))
try:
    _delete_all_collections(client)
except Exception as e:
    print(f"Post-recreate cleanup warning: {e}")

# 4) Recreate main chunk collection with sanitized metadata.
collection = client.get_or_create_collection(
    name="course_chunks",
    embedding_function=embedding_fn,
)

chunk_ids = df["id"].astype(str).tolist() if "id" in df.columns else df.index.astype(str).tolist()
sanitized_metadatas = [_sanitize_for_chroma_metadata(m) for m in df["metadata"].tolist()]
collection.upsert(
    ids=chunk_ids,
    documents=df["text"].astype(str).tolist(),
    metadatas=sanitized_metadatas,
)

# 5) Rebuild router indexes only if function has already been defined.
if "build_vector_router_indexes" in globals():
    build_vector_router_indexes(refresh=True)
    print("Router indexes rebuilt.")
else:
    print("Router index rebuild skipped for now. Run Cell 11, then rerun this cell if needed.")

print("Vector DB reset complete.")
print("Main collection count:", collection.count())

Folder cleanup warning: [WinError 32] The process cannot access the file because it is being used by another process: '..\\vectordb\\chroma_db\\2505871d-4348-4ff0-a667-0c81f58625e5\\data_level0.bin'
Router index rebuild skipped for now. Run Cell 11, then rerun this cell if needed.
Vector DB reset complete.
Main collection count: 609


In [20]:
df["course"].unique().tolist()

['Classification metrics',
 'Clustering metrics',
 'Conceptual   relational model',
 'Data cleaning',
 'Data visualization using python',
 'Datastructure in python',
 'Decision Tree',
 'EDA using pandas',
 'Feature Scaling',
 'File Handling',
 'Forward and Backward propagation',
 'function in python',
 'Gathering Data using API',
 'git cheat sheet',
 'Git',
 'Introduction to datascience',
 'Introduction to Statistics',
 'KNN',
 'Linear Regression',
 'Loss functions',
 'Numpy',
 'OOP in python',
 'Optimization algorithm',
 'Pandans intro (how to select and use loc and iloc)',
 'Python intro',
 'Regression metrics',
 'Train test split and cross validation',
 'Web Scraping']

In [22]:
(
	df.groupby("course", as_index=False)["part"]
	.agg(lambda s: sorted(pd.unique(s.dropna().astype(str)).tolist()))
	.rename(columns={"part": "parts"})
	.to_json(path_or_buf="course_parts.json", orient="records", force_ascii=False, indent=2)
)

In [23]:
import os
from pathlib import Path
from dotenv import load_dotenv

current = Path.cwd().resolve()
env_file = None
for _ in range(8):
    candidate = current / ".env"
    if candidate.exists():
        env_file = candidate
        break
    if current.parent == current:
        break
    current = current.parent

if env_file is not None:
    load_dotenv(dotenv_path=env_file, override=False)

hf_token = os.getenv("HF_TOKEN") or os.getenv("HUGGINGFACEHUB_API_TOKEN")

In [24]:
from huggingface_hub import login
login(hf_token)

The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `huggingface-cli` if you want to set the git credential as well.
Token is valid (permission: fineGrained).
Your token has been saved to C:\Users\melki\.cache\huggingface\token
Login successful


In [31]:
import hashlib
import json
import os
from typing import Dict, List, Any, Optional

try:
    from scipy.spatial import distance as scipy_distance
except Exception:
    scipy_distance = None

from sentence_transformers import CrossEncoder

# Vector-router collections (separate from chunk collection).
COURSE_COLLECTION_NAME = "course_profiles_index"
PART_COLLECTION_NAME = "course_parts_profiles_index"
RERANKER_MODEL_NAME = "BAAI/bge-reranker-large"

HF_TOKEN = str(globals().get("hf_token", "") or "").strip()
if not HF_TOKEN:
    raise RuntimeError("HF token not found. Run the hf_token cell or add HF_TOKEN to your project .env file.")


ROUTER_INDEX_SIGNATURE: Optional[str] = None
ROUTER_INDEX_READY = False
all_course_values: List[str] = []
cross_encoder_reranker: Optional[CrossEncoder] = None


def _normalize_text(text: str) -> str:
    return " ".join(str(text).split())


def _extract_query_field(result: Any, key: str) -> Any:
    if isinstance(result, dict):
        return result.get(key)
    return getattr(result, key, None)


def _flatten_query_field(raw_value: Any) -> List[Any]:
    if isinstance(raw_value, list):
        if raw_value and isinstance(raw_value[0], list):
            return raw_value[0]
        return raw_value
    return []


def _similarity_from_result(
    query_embedding: Optional[List[float]],
    candidate_embedding: Any,
    db_similarity: Any,
    db_score: Any,
    db_distance: Any,
) -> float:
    if isinstance(db_similarity, (int, float)):
        return float(db_similarity)
    if isinstance(db_score, (int, float)):
        return float(db_score)

    if (
        scipy_distance is not None
        and isinstance(query_embedding, list)
        and query_embedding
        and isinstance(candidate_embedding, list)
        and candidate_embedding
    ):
        try:
            return 1.0 - float(scipy_distance.cosine(query_embedding, candidate_embedding))
        except Exception:
            pass

    if isinstance(db_distance, (int, float)):
        return -float(db_distance)

    return float("-inf")


def _query_records(res: Any, query_embedding: Optional[List[float]] = None) -> List[Dict[str, Any]]:
    if res is None:
        return []

    ids = _flatten_query_field(_extract_query_field(res, "ids"))
    documents = _flatten_query_field(_extract_query_field(res, "documents"))
    metadatas = _flatten_query_field(_extract_query_field(res, "metadatas"))
    distances = _flatten_query_field(_extract_query_field(res, "distances"))
    similarities = _flatten_query_field(_extract_query_field(res, "similarities"))
    scores = _flatten_query_field(_extract_query_field(res, "scores"))
    embeddings = _flatten_query_field(_extract_query_field(res, "embeddings"))

    n = max(
        len(ids),
        len(documents),
        len(metadatas),
        len(distances),
        len(similarities),
        len(scores),
        len(embeddings),
    )

    return [
        {
            "id": str(ids[i]) if i < len(ids) else "",
            "document": str(documents[i]) if i < len(documents) else "",
            "metadata": metadatas[i] if i < len(metadatas) and isinstance(metadatas[i], dict) else {},
            "distance": float(distances[i]) if i < len(distances) and isinstance(distances[i], (int, float)) else None,
            "similarity": _similarity_from_result(
                query_embedding=query_embedding,
                candidate_embedding=embeddings[i] if i < len(embeddings) else None,
                db_similarity=similarities[i] if i < len(similarities) else None,
                db_score=scores[i] if i < len(scores) else None,
                db_distance=distances[i] if i < len(distances) else None,
            ),
        }
        for i in range(n)
    ]


def _is_generic_part(part: str) -> bool:
    return str(part).strip().lower() in {"full document", "document", "full"}


def get_cross_encoder() -> CrossEncoder:
    global cross_encoder_reranker

    if "cross_encoder_reranker" not in globals() or cross_encoder_reranker is None:
        try:
            cross_encoder_reranker = CrossEncoder(
                RERANKER_MODEL_NAME
            )
        except TypeError:
            cross_encoder_reranker = CrossEncoder(RERANKER_MODEL_NAME)

    return cross_encoder_reranker


def _router_data_signature(frame: Any) -> str:
    if frame is None or getattr(frame, "empty", True):
        return "empty"

    subset = frame[["course", "part", "text"]].dropna(subset=["course", "part", "text"]).astype(str)
    if subset.empty:
        return "empty"

    subset = subset.sort_values(["course", "part", "text"], kind="mergesort")
    hasher = hashlib.sha256()
    for row in subset.itertuples(index=False, name=None):
        hasher.update("\x1f".join(row).encode("utf-8", errors="ignore"))
        hasher.update(b"\n")
    return hasher.hexdigest()


# -------------------------
# Hierarchical filtering
# -------------------------
def build_vector_router_indexes(
    refresh: bool = False,
    course_samples_per_course: int = 30,
    part_samples_per_pair: int = 10,
    profile_char_limit: int = 4000,
) -> Dict[str, Any]:
    global course_collection, part_collection, all_course_values
    global ROUTER_INDEX_SIGNATURE, ROUTER_INDEX_READY

    if "df" not in globals():
        raise RuntimeError("df is not available. Run the dataframe preparation cells first.")

    current_signature = _router_data_signature(df)
    should_rebuild = bool(refresh) or (not ROUTER_INDEX_READY) or (ROUTER_INDEX_SIGNATURE != current_signature)

    if not should_rebuild and "course_collection" in globals() and "part_collection" in globals():
        return {
            "rebuilt": False,
            "signature": ROUTER_INDEX_SIGNATURE,
            "courses_indexed": len(all_course_values),
        }

    for name in [COURSE_COLLECTION_NAME, PART_COLLECTION_NAME]:
        try:
            client.delete_collection(name=name)
        except Exception:
            pass

    course_collection = client.get_or_create_collection(
        name=COURSE_COLLECTION_NAME,
        embedding_function=embedding_fn,
    )
    part_collection = client.get_or_create_collection(
        name=PART_COLLECTION_NAME,
        embedding_function=embedding_fn,
    )

    course_docs: List[str] = []
    course_meta: List[Dict[str, Any]] = []

    for course, group in df.groupby("course", dropna=True, sort=True):
        group = group.dropna(subset=["text", "part"])
        if group.empty:
            continue

        parts = sorted(group["part"].astype(str).drop_duplicates().tolist())
        sample_n = min(course_samples_per_course, len(group))
        sampled = group.sort_values(["part", "text"], kind="mergesort").head(sample_n)
        snippets = [_normalize_text(t)[:220] for t in sampled["text"].astype(str).tolist()]

        profile = (
            f"course: {course}\n"
            f"parts: {' | '.join(parts[:40])}\n"
            f"content: {' '.join(snippets)}"
        )[:profile_char_limit]

        course_docs.append(profile)
        course_meta.append({"course": str(course), "parts_count": len(parts)})

    all_course_values = [m["course"] for m in course_meta]
    if course_docs:
        course_collection.upsert(
            ids=[f"course::{i}" for i in range(len(course_docs))],
            documents=course_docs,
            metadatas=course_meta,
        )

    part_docs: List[str] = []
    part_meta: List[Dict[str, Any]] = []
    part_pairs = df[["course", "part", "text"]].dropna(subset=["course", "part", "text"])

    for (course, part), group in part_pairs.groupby(["course", "part"], dropna=True, sort=True):
        sample_n = min(part_samples_per_pair, len(group))
        sampled = group.sort_values(["text"], kind="mergesort").head(sample_n)
        snippets = [_normalize_text(t)[:220] for t in sampled["text"].astype(str).tolist()]

        profile = (
            f"course: {course}\n"
            f"part: {part}\n"
            f"content: {' '.join(snippets)}"
        )[:profile_char_limit]

        part_docs.append(profile)
        part_meta.append({"course": str(course), "part": str(part)})

    if part_docs:
        part_collection.upsert(
            ids=[f"part::{i}" for i in range(len(part_docs))],
            documents=part_docs,
            metadatas=part_meta,
        )

    ROUTER_INDEX_SIGNATURE = current_signature
    ROUTER_INDEX_READY = True

    return {
        "rebuilt": True,
        "signature": ROUTER_INDEX_SIGNATURE,
        "courses_indexed": len(course_docs),
        "parts_indexed": len(part_docs),
    }


def retrieve_courses(
    query: str,
    desired_courses: int = 2,
    max_candidates: int = 8,
) -> List[str]:
    if not ROUTER_INDEX_READY:
        build_vector_router_indexes(refresh=False)

    if not all_course_values:
        return []

    query_embedding = embed_query(query)
    k = min(max(max_candidates, desired_courses), len(all_course_values))
    res = course_collection.query(
        query_embeddings=[query_embedding],
        n_results=k,
        include=["metadatas", "distances", "embeddings"],
    )

    best_by_course: Dict[str, float] = {}
    for rec in _query_records(res, query_embedding=query_embedding):
        course = str(rec["metadata"].get("course", "")).strip()
        if not course:
            continue

        score = float(rec.get("similarity", float("-inf")))
        prev = best_by_course.get(course)
        if prev is None or score > prev:
            best_by_course[course] = score

    ranked = sorted(best_by_course.items(), key=lambda x: (-x[1], x[0]))
    return [course for course, _ in ranked[: max(1, desired_courses)]]


def retrieve_parts_for_courses(
    query: str,
    courses: List[str],
    parts_per_course: int = 2,
    candidate_multiplier: int = 6,
    min_per_course: int = 1,
) -> Dict[str, List[str]]:
    if not ROUTER_INDEX_READY:
        build_vector_router_indexes(refresh=False)

    mapping: Dict[str, List[str]] = {}
    query_embedding = embed_query(query)

    for course in courses:
        n_candidates = max(parts_per_course * candidate_multiplier, min_per_course)
        res = part_collection.query(
            query_embeddings=[query_embedding],
            n_results=n_candidates,
            where={"course": course},
            include=["metadatas", "distances", "embeddings"],
        )

        best_by_part: Dict[str, float] = {}
        for rec in _query_records(res, query_embedding=query_embedding):
            part = str(rec["metadata"].get("part", "")).strip()
            if not part:
                continue

            score = float(rec.get("similarity", float("-inf")))
            prev = best_by_part.get(part)
            if prev is None or score > prev:
                best_by_part[part] = score

        ranked = sorted(best_by_part.items(), key=lambda x: (-x[1], x[0]))
        kept = [part for part, _ in ranked[: max(parts_per_course, min_per_course)]]

        non_generic = [p for p in kept if not _is_generic_part(p)]
        if non_generic:
            kept = non_generic

        if kept:
            mapping[course] = kept[: max(parts_per_course, min_per_course)]

    return mapping


def route_query_to_course_part(
    query: str,
    n_courses: int = 2,
    parts_per_course: int = 2,
    candidate_multiplier: int = 6,
    course_candidates: int = 8,
) -> Dict[str, List[str]]:
    courses = retrieve_courses(
        query=query,
        desired_courses=n_courses,
        max_candidates=course_candidates,
    )
    if not courses:
        return {}

    mapping = retrieve_parts_for_courses(
        query=query,
        courses=courses,
        parts_per_course=parts_per_course,
        candidate_multiplier=candidate_multiplier,
    )

    return {c: parts for c, parts in mapping.items() if parts}


# -------------------------
# Target architecture pipeline
# User Question -> Query embedding -> Vector search -> Hierarchical filtering
# -> Neural reranking -> Context compression -> LLM answer generation -> Source attribution
# -------------------------
def embed_query(query: str) -> List[float]:
    return embed_text(query)


def vector_search(query_embedding: List[float], n_results: int = 120) -> List[Dict[str, Any]]:
    res = collection.query(
        query_embeddings=[query_embedding],
        n_results=n_results,
        include=["documents", "metadatas", "distances"],
    )
    return _query_records(res, query_embedding=query_embedding)


def hierarchical_filter(
    candidates: List[Dict[str, Any]],
    mapping: Dict[str, List[str]],
    max_after_filter: int = 48,
) -> List[Dict[str, Any]]:
    if not candidates:
        return []

    if not mapping:
        return candidates[:max_after_filter]

    allowed = {
        (str(course).strip(), str(part).strip())
        for course, parts in mapping.items()
        for part in parts
    }

    filtered = []
    for cand in candidates:
        meta = cand.get("metadata", {}) or {}
        key = (str(meta.get("course", "")).strip(), str(meta.get("part", "")).strip())
        if key in allowed:
            filtered.append(cand)

    return filtered[:max_after_filter] if filtered else candidates[:max_after_filter]


def neural_rerank(
    query: str,
    candidates: List[Dict[str, Any]],
    top_k: int = 12,
    rerank_pool: int = 40,
    max_doc_chars: int = 1800,
) -> List[Dict[str, Any]]:
    if not candidates:
        return []

    pool = candidates[: max(top_k, rerank_pool)]
    pairs = [(query, _normalize_text(cand.get("document", ""))[:max_doc_chars]) for cand in pool]

    reranker = get_cross_encoder()
    scores = reranker.predict(pairs, show_progress_bar=False)

    scored: List[Dict[str, Any]] = []
    for cand, score in zip(pool, scores):
        rec = dict(cand)
        rec["rerank_score"] = float(score)
        scored.append(rec)

    scored.sort(key=lambda x: x["rerank_score"], reverse=True)
    return scored[:top_k]


def _estimate_tokens(text: str) -> int:
    return max(1, len(str(text)) // 4)


def _safe_json_array(raw_text: str) -> List[Dict[str, Any]]:
    text = str(raw_text or "").strip()
    start = text.find("[")
    end = text.rfind("]")
    if start == -1 or end == -1 or end <= start:
        return []

    try:
        parsed = json.loads(text[start : end + 1])
    except json.JSONDecodeError:
        return []

    if not isinstance(parsed, list):
        return []

    return [row for row in parsed if isinstance(row, dict)]


def _llm_compress_context_items(
    question: str,
    items: List[Dict[str, Any]],
    target_tokens_per_source: int = 90,
) -> Dict[str, str]:
    if not items:
        return {}

    source_blocks = [
        (
            f"source_id: {item['source_id']}\n"
            f"course: {item['course']}\n"
            f"part: {item['part']}\n"
            f"text: {item['text']}"
        )
        for item in items
    ]

    prompt = (
        "You are compressing context for a RAG pipeline.\n"
        "Summarize each source independently.\n"
        "Preserve main ideas, formulas, equations, units, and technical constraints exactly.\n"
        "Do not add new facts.\n"
        f"Each summary must be <= {target_tokens_per_source} tokens.\n"
        "Return ONLY valid JSON as a list of objects with keys source_id and text.\n\n"
        f"Question:\n{question}\n\n"
        "Sources:\n"
        + "\n\n".join(source_blocks)
    )

    raw = llm_response(prompt)
    parsed = _safe_json_array(raw)

    return {
        str(item.get("source_id", "")).strip(): _normalize_text(item.get("text", ""))
        for item in parsed
        if str(item.get("source_id", "")).strip() and str(item.get("text", "")).strip()
    }


def context_compression(
    reranked: List[Dict[str, Any]],
    question: str,
    max_chunks: int = 6,
    compression_trigger_tokens: int = 1200,
    compressed_tokens_per_chunk: int = 90,
    fallback_max_chars_per_chunk: int = 650,
) -> List[Dict[str, Any]]:
    prepared: List[Dict[str, Any]] = []

    for rank, rec in enumerate(reranked[:max_chunks], start=1):
        meta = rec.get("metadata", {}) or {}
        page_start = meta.get("page_start")
        page_end = meta.get("page_end")

        if page_start is None and page_end is None:
            pages = "unknown"
        elif page_end is None or page_start == page_end:
            pages = str(page_start)
        else:
            pages = f"{page_start}-{page_end}"

        prepared.append(
            {
                "source_id": f"S{rank}",
                "chunk_id": rec.get("id", ""),
                "course": str(meta.get("course", "")),
                "part": str(meta.get("part", "")),
                "header_path": str(meta.get("header_path", "")),
                "source": str(meta.get("source", "")),
                "pages": pages,
                "score": round(float(rec.get("rerank_score", 0.0)), 4),
                "text": _normalize_text(rec.get("document", "")),
            }
        )

    if not prepared:
        return []

    total_tokens = sum(_estimate_tokens(item["text"]) for item in prepared)
    if total_tokens <= compression_trigger_tokens:
        return prepared

    summary_map = _llm_compress_context_items(
        question=question,
        items=prepared,
        target_tokens_per_source=compressed_tokens_per_chunk,
    )

    compressed: List[Dict[str, Any]] = []
    for item in prepared:
        summary_text = _normalize_text(summary_map.get(item["source_id"], ""))
        if summary_text:
            item["text"] = summary_text
        else:
            fallback = item["text"][:fallback_max_chars_per_chunk]
            if len(item["text"]) > fallback_max_chars_per_chunk:
                fallback = fallback + "..."
            item["text"] = fallback
        compressed.append(item)

    return compressed


def _format_context_for_prompt(items: List[Dict[str, Any]]) -> str:
    lines = []
    for item in items:
        lines.append(
            f"[{item['source_id']}] course={item['course']} | part={item['part']} | pages={item['pages']} | {item['text']}"
        )
    return "\n\n".join(lines) if lines else ""


def _log_context_snapshot(
    items: List[Dict[str, Any]],
    stage: str,
    max_items: int = 5,
    max_chars: int = 220,
) -> None:
    print(f"\n[{stage}] total_items={len(items)}")
    if not items:
        return

    for idx, rec in enumerate(items[:max_items], start=1):
        meta = rec.get("metadata", {}) or {}
        course = str(meta.get("course", ""))
        part = str(meta.get("part", ""))

        score_val = rec.get("rerank_score")
        if score_val is None:
            score_val = rec.get("similarity")
        score_str = f"{float(score_val):.4f}" if isinstance(score_val, (int, float)) else "n/a"

        text = _normalize_text(rec.get("document", ""))
        preview = text[:max_chars] + ("..." if len(text) > max_chars else "")

        print(f"  {idx}. course={course} | part={part} | score={score_str}")
        print(f"     text={preview}")


def generate_answer_with_sources(
    question: str,
    n_courses: int = 2,
    parts_per_course: int = 2,
    vector_k: int = 120,
    max_chunks_for_answer: int = 6,
    debug_log_context: bool = False,
    debug_log_max_items: int = 5,
) -> Dict[str, Any]:
    # Stage 1: Query embedding
    query_embedding = embed_query(question)

    # Stage 2: Vector search
    candidates = vector_search(query_embedding=query_embedding, n_results=vector_k)

    # Stage 3: Hierarchical filtering
    routing = route_query_to_course_part(
        query=question,
        n_courses=n_courses,
        parts_per_course=parts_per_course,
        candidate_multiplier=6,
        course_candidates=10,
    )
    filtered = hierarchical_filter(candidates, routing, max_after_filter=48)

    if debug_log_context:
        _log_context_snapshot(
            items=filtered,
            stage="Context Before Reranking",
            max_items=debug_log_max_items,
        )

    # Stage 4: Neural reranking (CrossEncoder)
    reranked = neural_rerank(question, filtered, top_k=12, rerank_pool=40)

    if debug_log_context:
        _log_context_snapshot(
            items=reranked,
            stage="Context After Reranking",
            max_items=debug_log_max_items,
        )

    # Stage 5: Context compression
    compressed = context_compression(
        reranked=reranked,
        question=question,
        max_chunks=max_chunks_for_answer,
        compression_trigger_tokens=1200,
        compressed_tokens_per_chunk=90,
        fallback_max_chars_per_chunk=650,
    )

    # Stage 6: LLM answer generation
    context_text = _format_context_for_prompt(compressed)
    prompt = (
        "You are a helpful assistant for course content Q&A. "
        "Use only the provided context to answer the question. "
        "If context is insufficient, say exactly: I do not have enough context to answer. "
        "Cite sources inline using tags like [S1], [S2].\n\n"
        f"Question:\n{question}\n\n"
        f"Context:\n{context_text}\n\n"
        "Answer:"
    )
    answer = llm_response(prompt)

    # Stage 7: Source attribution
    sources = [
        {
            "source_id": item["source_id"],
            "chunk_id": item["chunk_id"],
            "course": item["course"],
            "part": item["part"],
            "pages": item["pages"],
            "source": item["source"],
            "score": item["score"],
        }
        for item in compressed
    ]

    return {
        "question": question,
        "routing": routing,
        "retrieval_counts": {
            "vector_candidates": len(candidates),
            "after_hierarchical_filter": len(filtered),
            "after_neural_rerank": len(reranked),
            "after_context_compression": len(compressed),
        },
        "compressed_context": compressed,
        "sources": sources,
        "answer": answer,
    }


# Build vector-router indexes once.
router_index_status = build_vector_router_indexes(refresh=False)
print(router_index_status)


# Demo
demo_query = "How do I remove duplicates in pandas and what is the difference between Git and GitHub?"
pipeline_output = generate_answer_with_sources(demo_query, debug_log_context=True)
pipeline_output["routing"]

{'rebuilt': True, 'signature': '4b7c91d44dc10e316eef5205b2f5dbce99b7278bc108419c8db53bc675e7ee1f', 'courses_indexed': 28, 'parts_indexed': 398}

[Context Before Reranking] total_items=4
  1. course=EDA using pandas | part=1. Handling Duplicates in Data | score=-0.1784
     text=1. Handling Duplicates in Data Duplicates in a dataset occur when identical data entries are input more than once, often due to repeated data entry or merging data from multiple sources. Such duplicates can skew analysis...
  2. course=EDA using pandas | part=1. Exploring and Cleaning Data: | score=-0.3299
     text=1. Exploring and Cleaning Data: For the exploration and cleaning phase, we will utilize a dataset that comprises several key attributes. These include the Year of Birth, Gender, and Ethnicity of individuals, along with m...
  3. course=function in python | part=3. When to use Lambda vs. def | score=-0.3545
     text=3. When to use Lambda vs. def Lambda: Choose lambda functions for simple, concise tas

c:\Users\melki\anaconda3\envs\rag_main\Lib\site-packages\huggingface_hub\file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(



[Context After Reranking] total_items=4
  1. course=EDA using pandas | part=1. Handling Duplicates in Data | score=0.9401
     text=1. Handling Duplicates in Data Duplicates in a dataset occur when identical data entries are input more than once, often due to repeated data entry or merging data from multiple sources. Such duplicates can skew analysis...
  2. course=EDA using pandas | part=1. Exploring and Cleaning Data: | score=0.0030
     text=1. Exploring and Cleaning Data: For the exploration and cleaning phase, we will utilize a dataset that comprises several key attributes. These include the Year of Birth, Gender, and Ethnicity of individuals, along with m...
  3. course=function in python | part=3. When to use Lambda vs. def | score=0.0001
     text=3. When to use Lambda vs. def Lambda: Choose lambda functions for simple, concise tasks that are typically executed in a single line and not reused frequently. Lambdas are best for one-off operations where you need a qui...
  4. cour

{'EDA using pandas': ['1. Handling Duplicates in Data',
  '1. Exploring and Cleaning Data:'],
 'function in python': ['3. When to use Lambda vs. def',
  '1. How Do You Create a Function?']}

In [27]:
pipeline_output["routing"] if "pipeline_output" in globals() else {}

{'EDA using pandas': ['1. Handling Duplicates in Data',
  '1. Exploring and Cleaning Data:'],
 'function in python': ['3. When to use Lambda vs. def',
  '1. How Do You Create a Function?']}

In [45]:
example_query_2 = "what is a the role of botlane in league of legends?"
example_output_2 = route_query_to_course_part(example_query_2)
example_output_2

{}

In [46]:
query = "what is a the role of botlane in league of legends?"
pipeline_preview = generate_answer_with_sources(
    question=query,
    n_courses=2,
    parts_per_course=2,
    vector_k=120,
    max_chunks_for_answer=6,
)

len(pipeline_preview["compressed_context"])

0

In [47]:
#final_query = query if "query" in globals() else "How do I remove duplicates in pandas?"

result = generate_answer_with_sources(
    question=query,
    n_courses=2,
    parts_per_course=2,
    vector_k=120,
    max_chunks_for_answer=6,
)

final_answer = result["answer"]
context = result["compressed_context"]
sources = result["sources"]

final_answer

"I don't know but feel free to ask me about python ml datascience"

In [48]:
# Better off-domain reply message override
OUT_OF_SCOPE_REPLY = (
    "That question is outside my scope right now. "
    "I can help with Python, machine learning, and data science topics. "
    "Try asking about pandas, NumPy, SQL, model training, or data cleaning."
)

DOMAIN_GUARD_SYSTEM_PROMPT = (
    "You are a strict domain classifier. Classify user questions as IN_DOMAIN only if "
    "they are about Python programming, data science, or machine learning. "
    "Reply with exactly one token: IN_DOMAIN or OUT_OF_DOMAIN."
)

FINAL_ANSWER_SYSTEM_PROMPT = (
    "You are an assistant specialized in Python, Data Science, and Machine Learning. "
    "If the user question is outside these domains, respond exactly with this sentence: "
    f"{OUT_OF_SCOPE_REPLY}"
)

print("Updated off-domain reply message.")

Updated off-domain reply message.


In [49]:
# Off-domain reply check
check_off = generate_answer_with_sources("Who won the FIFA world cup in 2022?")
print(check_off.get("answer", ""))

That question is outside my scope right now. I can help with Python, machine learning, and data science topics. Try asking about pandas, NumPy, SQL, model training, or data cleaning.


In [52]:
# LangChain chain-based RAG pipeline (replaces block-style orchestration)
from chromadb import PersistentClient
from chromadb.utils import embedding_functions
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

if "OUT_OF_SCOPE_REPLY" not in globals():
    OUT_OF_SCOPE_REPLY = (
        "That question is outside my scope right now. "
        "I can help with Python, machine learning, and data science topics. "
        "Try asking about pandas, NumPy, SQL, model training, or data cleaning."
    )

if "DOMAIN_GUARD_SYSTEM_PROMPT" not in globals():
    DOMAIN_GUARD_SYSTEM_PROMPT = (
        "You are a strict domain classifier. Classify user questions as IN_DOMAIN only if "
        "they are about Python programming, data science, or machine learning. "
        "Reply with exactly one token: IN_DOMAIN or OUT_OF_DOMAIN."
    )

if "FINAL_ANSWER_SYSTEM_PROMPT" not in globals():
    FINAL_ANSWER_SYSTEM_PROMPT = (
        "You are an assistant specialized in Python, Data Science, and Machine Learning. "
        "If the user question is outside these domains, respond exactly with this sentence: "
        f"{OUT_OF_SCOPE_REPLY}"
    )

CHAIN_CHROMA_PATH = "../vectordb/chroma_db"
CHAIN_MAIN_COLLECTION = "course_chunks"


def _ensure_collection_handles(force_router_rebuild: bool = False) -> None:
    global client, embedding_fn, collection, ROUTER_INDEX_READY

    if "embedding_fn" not in globals() or embedding_fn is None:
        embedding_fn = embedding_functions.OllamaEmbeddingFunction(
            model_name=EMBEDDING_MODEL,
            url="http://localhost:11434/api/embeddings",
        )

    if "client" not in globals() or client is None:
        client = PersistentClient(path=CHAIN_CHROMA_PATH)

    collection = client.get_or_create_collection(
        name=CHAIN_MAIN_COLLECTION,
        embedding_function=embedding_fn,
    )

    if force_router_rebuild:
        ROUTER_INDEX_READY = False
        build_vector_router_indexes(refresh=True)


def _parse_ollama_chat_response(response: Any) -> str:
    if isinstance(response, dict):
        message = response.get("message", {})
        if isinstance(message, dict):
            return str(message.get("content", "") or "").strip()

    if hasattr(response, "message"):
        message_obj = getattr(response, "message")
        if isinstance(message_obj, dict):
            return str(message_obj.get("content", "") or "").strip()
        if hasattr(message_obj, "content"):
            return str(getattr(message_obj, "content") or "").strip()

    return str(response).strip()


def _invoke_ollama_from_prompt_value(prompt_value: Any) -> str:
    messages = []
    for msg in prompt_value.to_messages():
        if msg.type == "system":
            role = "system"
        elif msg.type in {"human", "user"}:
            role = "user"
        elif msg.type == "ai":
            role = "assistant"
        else:
            role = "user"
        messages.append({"role": role, "content": str(msg.content)})

    response = ollama.chat(model=LLM_MODEL, messages=messages)
    return _parse_ollama_chat_response(response)


def _normalize_domain_label(raw_label: str) -> str:
    text = str(raw_label).upper()
    if "OUT_OF_DOMAIN" in text:
        return "OUT_OF_DOMAIN"
    if "IN_DOMAIN" in text:
        return "IN_DOMAIN"
    return "OUT_OF_DOMAIN"


domain_guard_chain = (
    ChatPromptTemplate.from_messages(
        [
            ("system", DOMAIN_GUARD_SYSTEM_PROMPT),
            ("human", "Question: {question}\nReturn only IN_DOMAIN or OUT_OF_DOMAIN."),
        ]
    )
    | RunnableLambda(_invoke_ollama_from_prompt_value)
    | RunnableLambda(_normalize_domain_label)
)

answer_chain = (
    ChatPromptTemplate.from_messages(
        [
            ("system", FINAL_ANSWER_SYSTEM_PROMPT),
            (
                "human",
                "Use only the context below to answer. If context is insufficient, say exactly: "
                "I do not have enough context to answer. Cite sources with tags like [S1], [S2].\n\n"
                "Question:\n{question}\n\n"
                "Context:\n{context_text}\n\n"
                "Answer:",
            ),
        ]
    )
    | RunnableLambda(_invoke_ollama_from_prompt_value)
    | StrOutputParser()
)


def _prepare_chain_state(inputs: Dict[str, Any]) -> Dict[str, Any]:
    question = str(inputs.get("question", "")).strip()
    n_courses = int(inputs.get("n_courses", 2))
    parts_per_course = int(inputs.get("parts_per_course", 2))
    vector_k = int(inputs.get("vector_k", 120))
    max_chunks_for_answer = int(inputs.get("max_chunks_for_answer", 6))
    debug_log_context = bool(inputs.get("debug_log_context", False))
    debug_log_max_items = int(inputs.get("debug_log_max_items", 5))
    domain_label = str(inputs.get("domain_label", "OUT_OF_DOMAIN")).upper()

    if not question:
        return {
            "done": True,
            "result": {
                "question": question,
                "routing": {},
                "retrieval_counts": {
                    "vector_candidates": 0,
                    "after_hierarchical_filter": 0,
                    "after_neural_rerank": 0,
                    "after_context_compression": 0,
                },
                "compressed_context": [],
                "sources": [],
                "answer": "",
            },
        }

    if domain_label == "OUT_OF_DOMAIN":
        return {
            "done": True,
            "result": {
                "question": question,
                "routing": {},
                "retrieval_counts": {
                    "vector_candidates": 0,
                    "after_hierarchical_filter": 0,
                    "after_neural_rerank": 0,
                    "after_context_compression": 0,
                },
                "compressed_context": [],
                "sources": [],
                "answer": OUT_OF_SCOPE_REPLY,
            },
        }

    def _run_retrieval_once() -> Dict[str, Any]:
        query_embedding = embed_query(question)
        candidates = vector_search(query_embedding=query_embedding, n_results=vector_k)
        routing = route_query_to_course_part(
            query=question,
            n_courses=n_courses,
            parts_per_course=parts_per_course,
            candidate_multiplier=6,
            course_candidates=10,
        )
        filtered = hierarchical_filter(candidates, routing, max_after_filter=48)

        if debug_log_context:
            _log_context_snapshot(
                items=filtered,
                stage="Context Before Reranking",
                max_items=debug_log_max_items,
            )

        reranked = neural_rerank(question, filtered, top_k=12, rerank_pool=40)

        if debug_log_context:
            _log_context_snapshot(
                items=reranked,
                stage="Context After Reranking",
                max_items=debug_log_max_items,
            )

        compressed = context_compression(
            reranked=reranked,
            question=question,
            max_chunks=max_chunks_for_answer,
            compression_trigger_tokens=1200,
            compressed_tokens_per_chunk=90,
            fallback_max_chars_per_chunk=650,
        )

        return {
            "candidates": candidates,
            "routing": routing,
            "filtered": filtered,
            "reranked": reranked,
            "compressed": compressed,
        }

    _ensure_collection_handles(force_router_rebuild=False)
    try:
        retrieval = _run_retrieval_once()
    except Exception as exc:
        if "does not exist" not in str(exc).lower():
            raise
        _ensure_collection_handles(force_router_rebuild=True)
        retrieval = _run_retrieval_once()

    compressed = retrieval["compressed"]
    context_text = _format_context_for_prompt(compressed)
    sources = [
        {
            "source_id": item["source_id"],
            "chunk_id": item["chunk_id"],
            "course": item["course"],
            "part": item["part"],
            "pages": item["pages"],
            "source": item["source"],
            "score": item["score"],
        }
        for item in compressed
    ]

    return {
        "done": False,
        "prompt_vars": {
            "question": question,
            "context_text": context_text,
        },
        "result_base": {
            "question": question,
            "routing": retrieval["routing"],
            "retrieval_counts": {
                "vector_candidates": len(retrieval["candidates"]),
                "after_hierarchical_filter": len(retrieval["filtered"]),
                "after_neural_rerank": len(retrieval["reranked"]),
                "after_context_compression": len(retrieval["compressed"]),
            },
            "compressed_context": retrieval["compressed"],
            "sources": sources,
            "answer": "",
        },
    }


def _finalize_chain_result(state: Dict[str, Any]) -> Dict[str, Any]:
    if bool(state.get("done")):
        return state.get("result", {})

    prompt_vars = state.get("prompt_vars", {})
    result_base = state.get("result_base", {})
    answer = answer_chain.invoke(prompt_vars)
    output = dict(result_base)
    output["answer"] = answer
    return output


rag_chain = (
    RunnableLambda(lambda inp: {**inp, "question": str(inp.get("question", "")).strip()})
    | RunnablePassthrough.assign(domain_label=domain_guard_chain)
    | RunnableLambda(_prepare_chain_state)
    | RunnableLambda(_finalize_chain_result)
)


def generate_answer_with_sources(
    question: str,
    n_courses: int = 2,
    parts_per_course: int = 2,
    vector_k: int = 120,
    max_chunks_for_answer: int = 6,
    debug_log_context: bool = False,
    debug_log_max_items: int = 5,
) -> Dict[str, Any]:
    return rag_chain.invoke(
        {
            "question": question,
            "n_courses": n_courses,
            "parts_per_course": parts_per_course,
            "vector_k": vector_k,
            "max_chunks_for_answer": max_chunks_for_answer,
            "debug_log_context": debug_log_context,
            "debug_log_max_items": debug_log_max_items,
        }
    )

print("LangChain runnable RAG chain loaded and generate_answer_with_sources is now chain-based.")

LangChain runnable RAG chain loaded and generate_answer_with_sources is now chain-based.


In [53]:
# Chain behavior check
chain_test_in = generate_answer_with_sources("What is a pandas dataframe?")
print("IN routing:", chain_test_in.get("routing", {}))
print("IN answer preview:", chain_test_in.get("answer", "")[:220])

chain_test_out = generate_answer_with_sources("Who won the FIFA world cup in 2022?")
print("OUT answer:", chain_test_out.get("answer", ""))

IN routing: {'EDA using pandas': ['4.1. Overview', '2. Info Function'], 'Gathering Data using API': ['1. Overview', '3. Principles of RESTful Services']}
IN answer preview: A pandas DataFrame is a two-dimensional size-mutable, potentially heterogeneous tabular data structure with labeled axes (rows and columns) [S1]. It consists of three principal components: the data, rows, and columns. A 
OUT answer: That question is outside my scope right now. I can help with Python, machine learning, and data science topics. Try asking about pandas, NumPy, SQL, model training, or data cleaning.


In [35]:
context

[{'source_id': 'S1',
  'chunk_id': 'Data cleaning::s2::c6',
  'course': 'Data cleaning',
  'part': '2. Navigating Common Data Quality Issues in Analysis and Interpretation',
  'header_path': '2. Navigating Common Data Quality Issues in Analysis and Interpretation',
  'source': 'C:\\Users\\melki\\OneDrive\\Desktop\\7ankalis\\PYTHON\\AI_LilMose3da\\data\\raw\\Data cleaning.pdf',
  'pages': '1-6',
  'score': 0.8778,
  'text': 'Duplicate records can skew analysis results and lead to incorrect conclusions. Deduplication involves identifying duplicate entries, removing them from the dataset, and identifying redundant observations.'},
 {'source_id': 'S2',
  'chunk_id': 'EDA using pandas::s14::c1',
  'course': 'EDA using pandas',
  'part': '1. Handling Duplicates in Data',
  'header_path': '1. Handling Duplicates in Data',
  'source': 'C:\\Users\\melki\\OneDrive\\Desktop\\7ankalis\\PYTHON\\AI_LilMose3da\\data\\raw\\EDA using pandas.pdf',
  'pages': '9-10',
  'score': 0.7103,
  'text': 'Duplica